# Giai đoạn 2: Tiền xử lý dữ liệu (Data Preprocessing)
**Dự án:** Phân tích dữ liệu thương mại điện tử Olist (Brazil, 2016–2018)


Dựa trên kết luận của Notebook 01, sổ tay này thực hiện:
1. Đọc dữ liệu gốc (`data/raw/`), giữ zip code ở dạng chuỗi
2. Xoá dòng trùng lặp
3. Ép kiểu thời gian, xử lý thiếu, xử lý ngoại lai (gắn cờ thay vì cắt)
4. Đồng bộ khoá giữa các bảng (loại bản ghi mồ côi)
5. Tạo đặc trưng mới (thời gian giao, trễ hạn, tháng mua...)
6. Kiểm định chất lượng và xuất 9 bảng sạch + bảng tổng hợp `olist_master.csv` ra `data/processed/`

In [1]:
import os
import numpy as np
import pandas as pd

RAW_DIR = "../data/raw/"
PRO_DIR = "../data/processed/"
os.makedirs(PRO_DIR, exist_ok=True)

file_names = {
    'orders': 'olist_orders_dataset.csv',
    'items': 'olist_order_items_dataset.csv',
    'payments': 'olist_order_payments_dataset.csv',
    'reviews': 'olist_order_reviews_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'customers': 'olist_customers_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'geo': 'olist_geolocation_dataset.csv',
    'category': 'product_category_name_translation.csv'
}
# Đọc zip code dạng chuỗi để không mất số 0 ở đầu
zip_cols = {'customers': 'customer_zip_code_prefix', 'sellers': 'seller_zip_code_prefix', 'geo': 'geolocation_zip_code_prefix'}
dfs = {name: pd.read_csv(os.path.join(RAW_DIR, f), dtype={zip_cols[name]: str} if name in zip_cols else None)
       for name, f in file_names.items()}
raw_rows = {n: len(d) for n, d in dfs.items()}

# ===== Bước 2: Xoá trùng lặp =====
for name in dfs:
    dfs[name] = dfs[name].drop_duplicates().reset_index(drop=True)
pd.Series({n: raw_rows[n] - len(d) for n, d in dfs.items()}, name='số dòng trùng đã xoá').to_frame()

,số dòng trùng đã xoá
orders,0
items,0
payments,0
reviews,0
products,0
customers,0
sellers,0
geo,261831
category,0


## Bước 3: Làm sạch từng bảng
- **orders:** ép kiểu thời gian, chỉ giữ đơn `delivered` có ngày giao hợp lệ
- **items:** gắn cờ ngoại lai giá (không cắt giá để doanh thu đúng thực tế)
- **payments:** loại `not_defined` / giá trị 0, sửa trả góp 0 kỳ → 1
- **reviews:** 1 review mới nhất cho mỗi đơn, điền khuyết bình luận
- **products:** sửa tên cột sai chính tả, điền khuyết danh mục & số đo
- **category:** bổ sung bản dịch còn thiếu
- **customers / sellers / geo:** chuẩn hoá zip, thành phố; gộp toạ độ theo zip

In [2]:
log = {}

# A. ORDERS
o = dfs['orders']
time_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
             'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in time_cols:
    o[col] = pd.to_datetime(o[col], errors='coerce')
n0 = len(o)
o = o[(o['order_status'] == 'delivered') & o['order_delivered_customer_date'].notna()]
o = o[o['order_delivered_customer_date'] >= o['order_purchase_timestamp']]
o['order_approved_at'] = o['order_approved_at'].fillna(o['order_purchase_timestamp'])
dfs['orders'] = o.reset_index(drop=True)
log['orders'] = n0 - len(o)

# B. ITEMS
it = dfs['items']
it['shipping_limit_date'] = pd.to_datetime(it['shipping_limit_date'])
Q1, Q3 = it['price'].quantile([0.25, 0.75])
price_upper = Q3 + 1.5 * (Q3 - Q1)
it['is_price_outlier'] = it['price'] > price_upper
it['item_total'] = it['price'] + it['freight_value']
log['items'] = 0

# C. PAYMENTS
p = dfs['payments']
n0 = len(p)
p = p[(p['payment_type'] != 'not_defined') & (p['payment_value'] > 0)].copy()
p['payment_installments'] = p['payment_installments'].clip(lower=1)
dfs['payments'] = p.reset_index(drop=True)
log['payments'] = n0 - len(p)

# D. REVIEWS
r = dfs['reviews']
for col in ['review_creation_date', 'review_answer_timestamp']:
    r[col] = pd.to_datetime(r[col])
n0 = len(r)
r = r.sort_values('review_answer_timestamp').drop_duplicates('order_id', keep='last').copy()
r['has_comment'] = r['review_comment_message'].notna()
r['review_comment_title'] = r['review_comment_title'].fillna('No Title')
r['review_comment_message'] = r['review_comment_message'].fillna('No Comment')
dfs['reviews'] = r.reset_index(drop=True)
log['reviews'] = n0 - len(r)

# E. PRODUCTS
pr = dfs['products'].rename(columns={'product_name_lenght': 'product_name_length',
                                     'product_description_lenght': 'product_description_length'})
pr['product_category_name'] = pr['product_category_name'].fillna('unknown')
pr['product_weight_g'] = pr['product_weight_g'].replace(0, np.nan)
num_cols = pr.select_dtypes(include='number').columns
pr[num_cols] = pr[num_cols].fillna(pr[num_cols].median())
pr['product_volume_cm3'] = pr['product_length_cm'] * pr['product_height_cm'] * pr['product_width_cm']
dfs['products'] = pr
log['products'] = 0

# F. CATEGORY
extra = pd.DataFrame({
    'product_category_name': ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos', 'unknown'],
    'product_category_name_english': ['pc_gamer', 'portable_kitchen_and_food_preparers', 'unknown'],
})
dfs['category'] = (pd.concat([dfs['category'], extra], ignore_index=True)
                   .drop_duplicates('product_category_name').reset_index(drop=True))
log['category'] = 0

# G. CUSTOMERS / SELLERS
for name, pre in [('customers', 'customer'), ('sellers', 'seller')]:
    d = dfs[name]
    d[f'{pre}_zip_code_prefix'] = d[f'{pre}_zip_code_prefix'].str.zfill(5)
    d[f'{pre}_city'] = d[f'{pre}_city'].str.strip().str.lower()
    d[f'{pre}_state'] = d[f'{pre}_state'].str.strip().str.upper()
    log[name] = 0

# H. GEOLOCATION: lọc khung lãnh thổ Brazil, gộp 1 toạ độ trung bình / zip
g = dfs['geo']
n0 = len(g)
g['geolocation_zip_code_prefix'] = g['geolocation_zip_code_prefix'].str.zfill(5)
g = g[g['geolocation_lat'].between(-33.75, 5.27) & g['geolocation_lng'].between(-73.99, -34.79)]
dfs['geo'] = (g.groupby('geolocation_zip_code_prefix', as_index=False)
                .agg(geolocation_lat=('geolocation_lat', 'mean'),
                     geolocation_lng=('geolocation_lng', 'mean'),
                     geolocation_city=('geolocation_city', 'first'),
                     geolocation_state=('geolocation_state', 'first')))
log['geo'] = n0 - len(dfs['geo'])

print(f"Ngưỡng ngoại lai giá (IQR) = {price_upper:.2f} BRL; số dòng gắn cờ: {it['is_price_outlier'].sum():,}")
pd.Series(log, name='số dòng bị loại / gộp ở Bước 3').to_frame()

Ngưỡng ngoại lai giá (IQR) = 277.40 BRL; số dòng gắn cờ: 8,427


,số dòng bị loại / gộp ở Bước 3
orders,2971
items,0
payments,9
reviews,551
products,0
category,0
customers,0
sellers,0
geo,719322


## Bước 4: Đồng bộ khoá giữa các bảng
Sau khi lọc orders, các bảng con (items, payments, reviews) và bảng danh mục (customers, products, sellers) phải được lọc theo để không còn bản ghi mồ côi.

In [3]:
before = {n: len(d) for n, d in dfs.items()}

o = dfs['orders']
o = o[o['order_id'].isin(dfs['items']['order_id'])
      & o['order_id'].isin(dfs['payments']['order_id'])
      & o['customer_id'].isin(dfs['customers']['customer_id'])].copy()
dfs['orders'] = o.reset_index(drop=True)

valid_orders = set(o['order_id'])
for name in ['items', 'payments', 'reviews']:
    dfs[name] = dfs[name][dfs[name]['order_id'].isin(valid_orders)].reset_index(drop=True)

dfs['customers'] = dfs['customers'][dfs['customers']['customer_id'].isin(o['customer_id'])].reset_index(drop=True)
dfs['products'] = dfs['products'][dfs['products']['product_id'].isin(dfs['items']['product_id'])].reset_index(drop=True)
dfs['sellers'] = dfs['sellers'][dfs['sellers']['seller_id'].isin(dfs['items']['seller_id'])].reset_index(drop=True)

pd.DataFrame({'trước': before, 'sau': {n: len(d) for n, d in dfs.items()}}).assign(loại=lambda x: x['trước'] - x['sau'])

,trước,sau,loại
orders,96470,96469,1
items,112650,110186,2464
payments,103877,100744,3133
reviews,98673,95823,2850
products,32951,32214,737
customers,99441,96469,2972
sellers,3095,2970,125
geo,19010,19010,0
category,74,74,0


## Bước 5: Tạo đặc trưng mới (Feature Engineering)
| Đặc trưng | Ý nghĩa |
|---|---|
| `delivery_days` | Số ngày từ lúc mua đến lúc khách nhận |
| `estimated_days` | Số ngày giao dự kiến |
| `delay_days`, `is_late` | Số ngày trễ so với dự kiến (> 0 là trễ) |
| `purchase_year / month / year_month / weekday / hour` | Thời điểm mua phục vụ phân tích xu hướng |

In [4]:
o = dfs['orders']
buy = o['order_purchase_timestamp']
o['delivery_days'] = ((o['order_delivered_customer_date'] - buy).dt.total_seconds() / 86400).round(2)
o['estimated_days'] = ((o['order_estimated_delivery_date'] - buy).dt.total_seconds() / 86400).round(2)
o['delay_days'] = (o['order_delivered_customer_date'].dt.normalize() - o['order_estimated_delivery_date']).dt.days
o['is_late'] = o['delay_days'] > 0
o['purchase_year'] = buy.dt.year
o['purchase_month'] = buy.dt.month
o['purchase_year_month'] = buy.dt.to_period('M').astype(str)
o['purchase_weekday'] = buy.dt.dayofweek
o['purchase_hour'] = buy.dt.hour

o[['delivery_days', 'estimated_days', 'delay_days']].describe().T.round(2)

C:\Users\thuon\AppData\Local\Temp\ipykernel_1120\1904912974.py:10: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  o['purchase_weekday'] = buy.dt.dayofweek


,count,mean,std,min,25%,50%,75%,max
delivery_days,96469.0,12.56,9.55,0.53,6.77,10.22,15.72,209.63
estimated_days,96469.0,23.74,8.76,2.01,18.33,23.23,28.41,155.14
delay_days,96469.0,-11.88,10.18,-147.00,-17.00,-12.00,-7.00,188.00


## Bước 6: Bảng tổng hợp & Kiểm định chất lượng
Bảng `olist_master` ở mức **1 dòng = 1 sản phẩm trong đơn**, ghép đủ thông tin đơn hàng, sản phẩm, danh mục (tiếng Anh), khách hàng, người bán, review và thanh toán. Sau đó chạy bộ kiểm tra tự động để chắc chắn dữ liệu sạch.

In [5]:
pay = dfs['payments']
# Phương thức chính = phương thức có giá trị thanh toán lớn nhất trong đơn
main_type = pay.sort_values('payment_value', ascending=False).drop_duplicates('order_id')[['order_id', 'payment_type']]
pay_agg = (pay.groupby('order_id', as_index=False)
              .agg(payment_total=('payment_value', 'sum'),
                   payment_installments=('payment_installments', 'max'),
                   n_payments=('payment_sequential', 'count'))
              .merge(main_type, on='order_id'))

master = (dfs['items']
          .merge(dfs['orders'], on='order_id', how='inner')
          .merge(dfs['products'], on='product_id', how='left')
          .merge(dfs['category'], on='product_category_name', how='left')
          .merge(dfs['customers'], on='customer_id', how='left')
          .merge(dfs['sellers'], on='seller_id', how='left')
          .merge(dfs['reviews'][['order_id', 'review_score', 'has_comment']], on='order_id', how='left')
          .merge(pay_agg, on='order_id', how='left'))
master['same_state'] = master['customer_state'] == master['seller_state']

it = dfs['items']
checks = {
    'orders.order_id duy nhất': dfs['orders']['order_id'].is_unique,
    'reviews: tối đa 1 review / đơn': dfs['reviews']['order_id'].is_unique,
    'items.order_id ⊂ orders': it['order_id'].isin(dfs['orders']['order_id']).all(),
    'payments.order_id ⊂ orders': dfs['payments']['order_id'].isin(dfs['orders']['order_id']).all(),
    'items.product_id ⊂ products': it['product_id'].isin(dfs['products']['product_id']).all(),
    'items.seller_id ⊂ sellers': it['seller_id'].isin(dfs['sellers']['seller_id']).all(),
    'orders.customer_id ⊂ customers': dfs['orders']['customer_id'].isin(dfs['customers']['customer_id']).all(),
    'products không còn giá trị thiếu': dfs['products'].isna().sum().sum() == 0,
    'mọi danh mục đều có bản dịch': dfs['products']['product_category_name'].isin(dfs['category']['product_category_name']).all(),
    'geo: mỗi zip 1 dòng': dfs['geo']['geolocation_zip_code_prefix'].is_unique,
    'zip code đủ 5 ký tự': dfs['customers']['customer_zip_code_prefix'].str.len().eq(5).all(),
    'price, freight >= 0': (it[['price', 'freight_value']] >= 0).all().all(),
    'delivery_days >= 0': (dfs['orders']['delivery_days'] >= 0).all(),
    'master: số dòng = items': len(master) == len(it),
}
result = pd.Series(checks, name='đạt').to_frame()
assert result['đạt'].all(), result[~result['đạt']]
print(f"master: {master.shape[0]:,} dòng x {master.shape[1]} cột")
result

master: 110,186 dòng x 49 cột


,đạt
orders.order_id duy nhất,True
reviews: tối đa 1 review / đơn,True
items.order_id ⊂ orders,True
payments.order_id ⊂ orders,True
items.product_id ⊂ products,True
items.seller_id ⊂ sellers,True
orders.customer_id ⊂ customers,True
products không còn giá trị thiếu,True
mọi danh mục đều có bản dịch,True
geo: mỗi zip 1 dòng,True


## Bước 7: Xuất dữ liệu sạch
Ghi đè 9 file `clean_*.csv` và thêm `olist_master.csv` vào `data/processed/` để dùng cho Notebook 03, 04, 05, SQL và Dashboard.

In [6]:
for name, df in dfs.items():
    df.to_csv(os.path.join(PRO_DIR, f"clean_{file_names[name]}"), index=False)
master.to_csv(os.path.join(PRO_DIR, "olist_master.csv"), index=False)

summary = pd.DataFrame({'raw': raw_rows, 'clean': {n: len(d) for n, d in dfs.items()}})
summary.loc['master'] = [np.nan, len(master)]
summary['giữ lại (%)'] = (summary['clean'] / summary['raw'] * 100).round(2)
print("Đã lưu 9 bảng sạch + olist_master.csv vào data/processed/")
summary

Đã lưu 9 bảng sạch + olist_master.csv vào data/processed/


,raw,clean,giữ lại (%)
orders,99441.0,96469,97.01
items,112650.0,110186,97.81
payments,103886.0,100744,96.98
reviews,99224.0,95823,96.57
products,32951.0,32214,97.76
customers,99441.0,96469,97.01
sellers,3095.0,2970,95.96
geo,1000163.0,19010,1.90
category,71.0,74,104.23
master,NaN,110186,NaN
